# Notebook 10 | Expected Value vs. Sample Mean

In [1]:
from foundations_of_probability_and_statistics.cars.horsepower_moments import conditional_horsepower_mean
from foundations_of_probability_and_statistics.cars.sample_from_car_distribution import sample_from_car_distribution

import pandas as pd

# show all rows of data frames and series per default
pd.set_option("display.max_rows", None)

## Expectation against estimation

Fix a brand $b$ and sample $X = (H \mid B = b)$: each draw is one horsepower value from that brand's conditional table. Two objects share a value but not a nature. The conditional expectation is the random variable $\mathbb{E}[H \mid B]$ -- a function of the brand; evaluated at the point $B = b$ it becomes the number

$$\mathbb{E}[H \mid B = b] = \sum_h h \, p(H = h \mid B = b).$$

The sample mean of $n$ draws $X_1, \dots, X_n$ estimates that number:

$$\bar{X}_n = \frac{1}{n} \sum_{i=1}^n X_i, \qquad \bar{X}_n^{(b)} \to \mathbb{E}[H \mid B = b].$$

Intuition: expectation weights values by model probabilities; the sample mean weights them by empirical frequencies. As frequencies settle toward probabilities, the estimate settles toward the truth -- the law of large numbers (notebook 12) makes this precise. Reach for the mean whenever a whole distribution must be compressed into one representative number.

## Derivation

1. Fix the brand $b$ and read the conditional $p(H = h \mid B = b)$ from the package.
2. Weight each horsepower value by its probability and sum: $\sum_h h \, p(h \mid b)$.
3. For VW: $100 \cdot 0.6 + 200 \cdot 0.3 + 300 \cdot 0.1 = 150$.
4. The sample mean replaces probabilities with empirical frequencies, so it converges to the same number as $n$ grows.

The characteristic mistake is treating one realized average as the expectation: $\bar{X}_n$ is random (it wiggles per sample) while $\mathbb{E}[H \mid B = b]$ is fixed. Only in the limit do they meet.

## Worked example (by hand)

$$\mathbb{E}[H \mid B = \text{VW}] = 100 \cdot 0.6 + 200 \cdot 0.3 + 300 \cdot 0.1 = 150,$$
$$\mathbb{E}[H \mid B = \text{Porsche}] = 395.$$

Note how the VW expectation sits below every value except the most likely one: probability mass at 100 hp drags the average down. By-hand mean of the explicit toy subsample $[100, 100, 200, 300]$ -- fixed constants, not read off any sample:

$$\bar{X}_4 = \frac{100 + 100 + 200 + 300}{4} = 175.$$

The toy mean overshoots the truth (175 vs 150) because four draws cannot represent the 0.6 mass at 100 hp faithfully -- estimation error made visible.

In [2]:
import math

# exact conditional means against the by-hand values
assert math.isclose(conditional_horsepower_mean("VW"), 100 * 0.6 + 200 * 0.3 + 300 * 0.1)
assert math.isclose(conditional_horsepower_mean("VW"), 150.0)
assert math.isclose(conditional_horsepower_mean("Porsche"), 395.0)

# by-hand mean of the explicit toy subsample [100, 100, 200, 300]
toy = [100, 100, 200, 300]
assert math.isclose(sum(toy) / len(toy), 175.0)
sum(toy) / len(toy)

175.0

## Generalization

One sample of 200,000 cars: the per-brand sample mean $\bar{X}_n^{(b)}$ converges to $\mathbb{E}[H \mid B = b]$ for every brand. Compare the remaining gaps across brands -- the noisier the conditional table, the slower the approach, a preview of variance (notebook 11) controlling the rate.

In [3]:
cars = sample_from_car_distribution(n_cars=200_000, random_state=42)

# per-brand sample means against the exact conditional expectations
sample_means = cars.groupby("brand")["horsepower"].mean()
for brand in ["VW", "Porsche", "Ferrari"]:
    assert math.isclose(sample_means[brand], conditional_horsepower_mean(brand), rel_tol=0.02)
sample_means

brand
Ferrari    500.172932
Porsche    394.564928
VW         150.150042
Name: horsepower, dtype: float64

## References

- Blitzstein, J. K., Hwang, J. (2019): "Introduction to Probability", 2nd ed., Chapman & Hall/CRC, chapter "Conditional Expectation", https://www.routledge.com/Introduction-to-Probability-Second-Edition/Blitzstein-Hwang/p/book/9781138369917 (free PDF: https://probabilitybook.net/).
- Wasserman, L. (2004): "All of Statistics: A Concise Course in Statistical Inference", Springer Texts in Statistics, chapter "Expectation", https://doi.org/10.1007/978-0-387-21736-9.
- scipy.stats.rv_discrete, https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rv_discrete.html.